In [ ]:
import numpy as np
import torch
from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.loggers import WandbLogger

from data import CPSDatasetModule
from model import VAEModule

seed_everything(42)

config = {
    "batch_size": 64,
    "channels": [16, 32, 64],
    "dim_input": 256,
    "kernel_size": 5,
    "Nfc": 256,
    "latent_dim": 64,
    "max_pooling_kernel": 2,
    "lr": 1e-4,
    "threshold": 0.0,
    "wBCE": 15,
    "beta_max": 0.1,
    "kl_warmup_epochs": 5,
    "num_channels": 3,
    "max_epochs": 30,
    "train_path": "/scratch/user/u.ae271721/dataset/Dataset/Training_Set/",
    "test_path": "/scratch/user/u.ae271721/dataset/Dataset/Test_All/",
    "train_split": "/scratch/user/u.ae271721/dataset/Training_validation_split/training.npz",
    "val_split": "/scratch/user/u.ae271721/dataset/Training_validation_split/validation.npz",
}

train_files = np.load(config["train_split"])["arr_0"]
val_files = np.load(config["val_split"])["arr_0"]

datamodule = CPSDatasetModule(
    file_path_training=config["train_path"],
    file_path_test=config["test_path"],
    train_files_list=train_files,
    val_files_list=val_files,
    batch_size=config["batch_size"],
)

model = VAEModule(
    channels=config["channels"],
    dim_input=config["dim_input"],
    kernel_size=config["kernel_size"],
    Nfc=config["Nfc"],
    latent_dim=config["latent_dim"],
    max_pooling_kernel=config["max_pooling_kernel"],
    lr=config["lr"],
    threshold=config["threshold"],
    wBCE=config["wBCE"],
    beta_max=config["beta_max"],
    kl_warmup_epochs=config["kl_warmup_epochs"],
    in_channels=config["num_channels"],
)

wandb_logger = WandbLogger(
    project="ISeeYoo",
    name="VAE-multichannel-cbam",
    tags=["multi-channel", "CBAM", "betaVAE"],
)
wandb_logger.log_hyperparams(config)

trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=config["max_epochs"],
    logger=wandb_logger,
)
wandb_logger.watch(model, log_graph=False)

trainer.fit(model, datamodule=datamodule)

In [ ]:
import torch
from pytorch_lightning import Trainer
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
import numpy as np
from model import VAEModule
from data import CPSDatasetModule

config = {
    "batch_size": 64,
    "channels": [16, 32, 64],
    "dim_input": 256,
    "kernel_size": 5,
    "Nfc": 256,
    "latent_dim": 64,
    "max_pooling_kernel": 2,
    "lr": 1e-4,
    "threshold": 0.0,
    "wBCE": 15,
    "beta_max": 0.1,
    "kl_warmup_epochs": 5,
    "num_channels": 3,
    "max_epochs": 30,
    "train_path": "/scratch/user/u.ae271721/dataset/Dataset/Training_Set/",
    "test_path": "/scratch/user/u.ae271721/dataset/Dataset/Test_All/",
    "train_split": "/scratch/user/u.ae271721/dataset/Training_validation_split/training.npz",
    "val_split": "/scratch/user/u.ae271721/dataset/Training_validation_split/validation.npz",
}

checkpoint_path = "/scratch/user/u.ae271721/image-based-network-traffic-anomaly-detection/ISeeYoo/am4tcsi0/checkpoints/epoch=29-step=135000.ckpt"  # Update this!
model = VAEModule.load_from_checkpoint(
    checkpoint_path=checkpoint_path,
    channels=config["channels"],
    dim_input=config["dim_input"],
    kernel_size=config["kernel_size"],
    Nfc=config["Nfc"],
    latent_dim=config["latent_dim"],
    max_pooling_kernel=config["max_pooling_kernel"],
    lr=config["lr"],
    threshold=config["threshold"],
    wBCE=config["wBCE"],
    beta_max=config["beta_max"],
    kl_warmup_epochs=config["kl_warmup_epochs"],
    in_channels=config["num_channels"],
)

trainer = Trainer(accelerator="gpu", devices=1)
test_metrics = trainer.test(model, dataloaders=datamodule.test_dataloader())
print("Lightning test metrics:", test_metrics)

z_clean = torch.cat(model.z_values_clean, dim=0).detach().cpu().numpy()
z_anom = torch.cat(model.z_values_anomalous, dim=0).detach().cpu().numpy()
recon_clean = torch.cat(model.clean_errors, dim=0).detach().cpu().numpy()
recon_anom = torch.cat(model.anomalous_errors, dim=0).detach().cpu().numpy()

clf = IsolationForest(contamination=0.02, random_state=0)
clf.fit(z_clean)
latent_scores_clean = -clf.decision_function(z_clean)
latent_scores_anom = -clf.decision_function(z_anom)


def minmax(x):
    x = np.asarray(x)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

combo_clean = 0.5 * minmax(recon_clean) + 0.5 * minmax(latent_scores_clean)
combo_anom = 0.5 * minmax(recon_anom) + 0.5 * minmax(latent_scores_anom)


y_true = np.concatenate([np.zeros_like(combo_clean), np.ones_like(combo_anom)])
scores = np.concatenate([combo_clean, combo_anom])

auroc = roc_auc_score(y_true, scores)
auprc = average_precision_score(y_true, scores)
print(f"Combined AUROC: {auroc:.4f}")
print(f"Combined AUPRC: {auprc:.4f}")

threshold = np.quantile(combo_clean, 0.95)
preds = (scores > threshold).astype(int)
precision = (preds[y_true == 1].mean() if np.any(y_true == 1) else 0.0)
recall = preds[y_true == 1].sum() / np.sum(y_true == 1)
print(f"Threshold (95th clean quantile): {threshold:.4f}")
print(f"Precision@threshold: {precision:.4f}")
print(f"Recall@threshold: {recall:.4f}")

prec_curve, rec_curve, thr = precision_recall_curve(y_true, scores)
print("PR curve points:", len(prec_curve))

In [ ]:
import torch
import numpy as np
from pytorch_lightning import Trainer
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix
import matplotlib.pyplot as plt
from model import VAEModule
from data import CPSDatasetModule


checkpoint_path = "/scratch/user/u.ae271721/image-based-network-traffic-anomaly-detection/ISeeYoo/am4tcsi0/checkpoints/epoch=29-step=135000.ckpt"  # Update this!


config = {
    "channels": [16, 32, 64],
    "dim_input": 256,
    "kernel_size": 5,
    "Nfc": 256,
    "latent_dim": 64,
    "max_pooling_kernel": 2,
    "lr": 1e-4,
    "threshold": 0.0,
    "wBCE": 15,
    "beta_max": 0.1,
    "kl_warmup_epochs": 5,
    "num_channels": 3,
}


model = VAEModule.load_from_checkpoint(
    checkpoint_path=checkpoint_path,
    channels=config["channels"],
    dim_input=config["dim_input"],
    kernel_size=config["kernel_size"],
    Nfc=config["Nfc"],
    latent_dim=config["latent_dim"],
    max_pooling_kernel=config["max_pooling_kernel"],
    lr=config["lr"],
    threshold=config["threshold"],
    wBCE=config["wBCE"],
    beta_max=config["beta_max"],
    kl_warmup_epochs=config["kl_warmup_epochs"],
    in_channels=config["num_channels"],
)

model.eval()

train_files = np.load(config.get("train_split", "/scratch/user/u.ae271721/dataset/Training_validation_split/training.npz"))["arr_0"]
val_files = np.load(config.get("val_split", "/scratch/user/u.ae271721/dataset/Training_validation_split/validation.npz"))["arr_0"]

datamodule = CPSDatasetModule(   
    file_path_training=config.get("train_path", "/scratch/user/u.ae271721/dataset/Dataset/Training_Set"),
    file_path_test=config.get("test_path", "/scratch/user/u.ae271721/dataset/Dataset/Test_All/"),
    train_files_list=train_files,
    val_files_list=val_files,
    batch_size=config.get("batch_size", 64),
)

trainer = Trainer(accelerator="gpu", devices=1)
test_metrics = trainer.test(model, dataloaders=datamodule.test_dataloader())
print("Lightning test metrics:", test_metrics)


z_clean = torch.cat(model.z_values_clean, dim=0).detach().cpu().numpy()
z_anom = torch.cat(model.z_values_anomalous, dim=0).detach().cpu().numpy()
recon_clean = torch.cat(model.clean_errors, dim=0).detach().cpu().numpy()
recon_anom = torch.cat(model.anomalous_errors, dim=0).detach().cpu().numpy()


clf = IsolationForest(contamination=0.02, random_state=0)
clf.fit(z_clean)
latent_scores_clean = -clf.decision_function(z_clean)
latent_scores_anom = -clf.decision_function(z_anom)


def minmax(x):
    x = np.asarray(x)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

combo_clean = 0.5 * minmax(recon_clean) + 0.5 * minmax(latent_scores_clean)
combo_anom = 0.5 * minmax(recon_anom) + 0.5 * minmax(latent_scores_anom)


y_true = np.concatenate([np.zeros_like(combo_clean), np.ones_like(combo_anom)])
scores = np.concatenate([combo_clean, combo_anom])


pfa_values = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
results = []

print("\n" + "="*80)
print("Testing on Multiple PFA (Probability of False Alarm) Values")
print("="*80)
print(f"{'PFA':<8} {'Threshold':<12} {'TPR':<8} {'Precision':<12} {'Recall':<10} {'F1':<8} {'TP':<6} {'FP':<6} {'TN':<6} {'FN':<6}")
print("-"*80)

for pfa in pfa_values:
    threshold = np.quantile(combo_clean, 1.0 - pfa)
    
    preds = (scores > threshold).astype(int)
    

    tn = np.sum((preds == 0) & (y_true == 0))
    fp = np.sum((preds == 1) & (y_true == 0))
    fn = np.sum((preds == 0) & (y_true == 1))
    tp = np.sum((preds == 1) & (y_true == 1))
    
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0  # Recall/TPR
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tpr
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    results.append({
        'pfa': pfa,
        'threshold': threshold,
        'tpr': tpr,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'tn': tn,
        'fn': fn
    })
    
    print(f"{pfa:<8.3f} {threshold:<12.6f} {tpr:<8.4f} {precision:<12.4f} {recall:<10.4f} {f1:<8.4f} {tp:<6} {fp:<6} {tn:<6} {fn:<6}")


auroc = roc_auc_score(y_true, scores)
auprc = average_precision_score(y_true, scores)
print("-"*80)
print(f"\nOverall AUROC: {auroc:.4f}")
print(f"Overall AUPRC: {auprc:.4f}")

try:
    from sklearn.metrics import roc_curve
    fpr, tpr_curve, _ = roc_curve(y_true, scores)
    prec_curve, rec_curve, _ = precision_recall_curve(y_true, scores)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax1.plot(fpr, tpr_curve, label=f'ROC (AUC = {auroc:.4f})')
    ax1.plot([0, 1], [0, 1], 'k--', label='Random')
    ax1.set_xlabel('False Positive Rate (PFA)')
    ax1.set_ylabel('True Positive Rate (TPR)')
    ax1.set_title('ROC Curve')
    ax1.legend()
    ax1.grid(True)
    
    
    ax2.plot(rec_curve, prec_curve, label=f'PR (AUC = {auprc:.4f})')
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title('Precision-Recall Curve')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not plot curves: {e}")
    
results_dict = {r['pfa']: r for r in results}
print("\nResults dictionary available as 'results_dict'")